# Akan (Asante Twi) TTS — Fine-tuning facebook/mms-tts-aka

**Author:** Abdul Rashid Dickson · **Task:** UG MSc stage-2 TTS assignment

Pipeline: prepared single-narrator corpus (~5–10 h Asante Twi, GhanaNLP *ghana-speech*, CC-BY-NC-4.0)
→ fine-tune Meta's `mms-tts-aka` VITS checkpoint with the `ylacombe/finetune-hf-vits` harness.

Runtime: **GPU T4** (Runtime → Change runtime type → T4 GPU). Checkpoints save to Google Drive,
so the run survives session limits and can be resumed.

In [ ]:
# 1. Install harness + build the Cython monotonic-align kernel (required for speed)
!git clone https://github.com/ylacombe/finetune-hf-vits
%cd finetune-hf-vits
!pip install -q -r requirements.txt
!pip install -q 'datasets[audio]' librosa soundfile accelerate
%cd monotonic_align
!mkdir -p monotonic_align && python setup.py build_ext --inplace
%cd ..

In [ ]:
# 2. Authenticate with Hugging Face (write token) — needed to pull the discriminator and push the model
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 3. Get the prepared corpus (zip of tts_data built by src/prepare_dataset.py)
# Either upload manually, or mount Drive:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q '/content/drive/MyDrive/UG_TTS/tts_data.zip' -d /content/
!ls /content/tts_data | head

In [ ]:
# 4. Build the TRAINING checkpoint for 'aka': generator (hub) + Meta's original Akan discriminator
!python convert_original_discriminator_checkpoint.py --language_code aka --pytorch_dump_folder_path /content/aka_train_base
!ls /content/aka_train_base

In [ ]:
# 5. Training config (hyperparameters tuned for ~10 h single-speaker data on T4)
import json, pathlib
cfg = {
  "project_name": "akan_twi_tts",
  "push_to_hub": True,
  "hub_model_id": "REPLACE_WITH_YOUR_HF_USERNAME/akan-twi-tts-mms",
  "report_to": [],            # wandb optional
  "overwrite_output_dir": True,
  "output_dir": "/content/akan-vits-finetuned",

  "dataset_name": "/content/tts_data",
  "dataset_config_name": None,
  "audio_column_name": "audio",
  "text_column_name": "text",
  "train_split_name": "train",
  "eval_split_name": "eval",
  "speaker_id_column_name": "speaker_id",
  "filter_on_speaker_id": 0,
  "override_speaker_embeddings": False,

  "full_generation_sample_text": "Mema wo akye, me nuanom. Wo ho te sɛn?",

  "max_duration_in_seconds": 12.0,
  "min_duration_in_seconds": 2.0,
  "max_tokens_length": 500,

  "model_name_or_path": "/content/aka_train_base",

  "preprocessing_num_workers": 2,
  "do_train": True,
  "num_train_epochs": 30,
  "gradient_accumulation_steps": 1,
  "per_device_train_batch_size": 16,
  "learning_rate": 2e-4,
  "lr_scheduler_type": "cosine",
  "warmup_ratio": 0.02,
  "adam_beta1": 0.8,
  "adam_beta2": 0.99,
  "weight_decay": 0.01,
  "group_by_length": False,

  "do_eval": True,
  "eval_steps": 500,
  "per_device_eval_batch_size": 8,
  "max_eval_samples": 25,
  "do_step_schedule_per_epoch": False,
  "save_steps": 1000,
  "save_total_limit": 3,

  "weight_disc": 3,
  "weight_fmaps": 1,
  "weight_gen": 1,
  "weight_kl": 1.5,
  "weight_duration": 1,
  "weight_mel": 35,

  "fp16": True,
  "seed": 456
}
pathlib.Path('/content/finetune_akan.json').write_text(json.dumps(cfg, indent=2))
print('config written')

In [ ]:
# 6. Train (T4: ~10k–20k steps; checkpoint every save_steps so you can resume)
# If a session dies mid-run: re-run this cell — resume_from_checkpoint picks up the latest.
!accelerate launch run_vits_finetuning.py /content/finetune_akan.json \
    --save_steps 1000 --save_total_limit 3 --resume_from_checkpoint true

In [ ]:
# 7. Quick listening test — unseen Twi text (NOT from the corpus)
import sys, torch, soundfile as sf
sys.path.insert(0, '/content')  # normalize_twi.py is in tts_data zip root
from normalize_twi import normalize_twi
from transformers import VitsModel, VitsTokenizer
M = '/content/akan-vits-finetuned'
tok = VitsTokenizer.from_pretrained(M)
model = VitsModel.from_pretrained(M).eval()
text = 'Mema wo akye, me nuanom. Wo ho te sɛn?'
inputs = tok(text=normalize_twi(text), return_tensors='pt')
with torch.no_grad():
    wav = model(**inputs).waveform[0]
sf.write('/content/demo_unseen.wav', (wav.numpy()*32767).astype('int16'), model.config.sampling_rate)
import IPython; IPython.display.Audio('/content/demo_unseen.wav')

In [ ]:
# 7b. ASR judge benchmark (round-trip evaluation judge selection)
# Whisper has no 'tw' language token -> use Akan/Twi fine-tuned checkpoints + English carrier prompt.
# Ground-truth clips: upload 06_samples/probe/*.wav to /content/probe/ with the truth dict below.
CANDIDATES = [
    'sadicko/whisper-small-akan-finetuned',
    'worldboss/whisper-small-twi-v1',
    'fiifinketia/whisper-large-v3-turbo-akan',
]
TRUTH = {
 'probe_0': '1 BERƐSOSƐM 1.',
 'probe_1': 'Ɛfiri Adam so de kɔsi Abraham so.',
 'probe_2': 'Adam, Set, Enos, Kenan, Mahalalel, Yared, Hanok, Metusela, Lamek, Noa, Sem, Ham ne Yafet.',
 'probe_3': 'Yafet mma ne: Gomer, Magog, Madai, Yawan, Tubal, Mesek ne Tiras.',
 'probe2_0': 'Ntonto a ɛtɔ so ɛnan no bɔɔ Seorim.',
 'probe2_1': 'Ntonto a ɛtɔ so enum no bɔɔ Malkia.',
 'probe2_2': 'Ntonto a ɛtɔ so nsia no bɔɔ Miyamin.',
}
import sys
sys.path.insert(0, '/content')
from normalize_twi import normalize_twi
from transformers import pipeline
import glob

def cer_(ref, hyp):
    r = list(normalize_twi(ref).replace(' ', '')); h = list(normalize_twi(hyp).replace(' ', ''))
    import editdistance
    return editdistance.eval(r, h) / max(1, len(r))

!pip install -q editdistance
results = {}
for cand in CANDIDATES:
    try:
        pipe = pipeline('automatic-speech-recognition', model=cand, device=0)
        cers = []
        for p in sorted(glob.glob('/content/probe/*.wav')):
            stem = p.split('/')[-1].replace('.wav','')
            hyp = pipe(p)['text'].strip()
            c = cer_(TRUTH.get(stem, ''), hyp)
            cers.append(c)
            print(cand, stem, round(c,2), '::', hyp[:60])
        results[cand] = sum(cers)/len(cers)
        del pipe
        import gc, torch; gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        print(cand, 'FAILED:', e)
print('\nRANKING (mean CER):'); [print(f'  {m}: {c}') for m, c in sorted(results.items(), key=lambda kv: kv[1])]

In [ ]:
# 8. Save checkpoint + samples to Drive (survives session loss; also push_to_hub from cfg)
!zip -qr /content/drive/MyDrive/UG_TTS/akan_tts_ckpt_$(date +%m%d).zip /content/akan-vits-finetuned
print('saved to Drive')